# Taller 5: Open Table Formats con Apache Iceberg

## Sistemas Intensivos en Datos

---

### Objetivos de aprendizaje

Al finalizar este taller, el estudiante sera capaz de:

1. **Explicar el problema** que tienen los archivos Parquet planos cuando se necesita ACID, historial de cambios o evolucion de schema.
2. **Crear una tabla Iceberg** sobre MinIO usando el catalogo REST de Iceberg.
3. **Ejecutar UPDATE y DELETE** transaccionales sobre una tabla Iceberg desde Spark SQL.
4. **Usar Time Travel** para leer el estado de la tabla en un snapshot anterior.
5. **Hacer Schema Evolution** agregando columnas sin reescribir los datos existentes.

---

### Contexto en la serie de talleres

| Taller | Tecnologia | Foco |
|--------|-----------|------|
| Taller 2 | HDFS + MapReduce | Almacenamiento distribuido + computo batch |
| Taller 3 | HDFS + Spark | Motor de procesamiento en memoria |
| Taller 4 | Spark + DataFrames + Parquet | Formato columnar eficiente |
| **Taller 5** | **MinIO + Spark + Iceberg** | **Open Table Formats: ACID, time travel, schema evolution** |
| Taller 6 | Lakehouse con Trino | Consultas SQL federadas sobre el lakehouse |

---

### Arquitectura de este taller

```
Jupyter Notebook
      |
      v
  Spark Engine  <------>  Iceberg REST Catalog
      |                        |
      v                        v
    MinIO (s3://warehouse/)   metadata JSON
      |
      +-- data/         (archivos Parquet reales)
      +-- metadata/     (snapshots, manifests, table metadata)
```

**MinIO** actua como object storage compatible con S3 — los datos viven como archivos Parquet pero Iceberg agrega una capa de metadatos que habilita las capacidades ACID.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = SparkSession.builder \
    .appName("Taller5-Iceberg") \
    .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions") \
    .config("spark.sql.catalog.demo", "org.apache.iceberg.spark.SparkCatalog") \
    .config("spark.sql.catalog.demo.type", "rest") \
    .config("spark.sql.catalog.demo.uri", "http://rest:8181") \
    .config("spark.sql.catalog.demo.io-impl", "org.apache.iceberg.aws.s3.S3FileIO") \
    .config("spark.sql.catalog.demo.warehouse", "s3://warehouse/") \
    .config("spark.sql.catalog.demo.s3.endpoint", "http://minio:9000") \
    .config("spark.sql.defaultCatalog", "demo") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")
print("SparkSession creada con catalogo Iceberg REST")
print(f"Version de Spark: {spark.version}")

## Parte 1: El problema con Parquet plano

En el **Taller 4** usamos Parquet directamente como formato de almacenamiento. Parquet es excelente para lecturas analíticas (columnar, compresion, predicado pushdown), pero tiene limitaciones fundamentales:

| Operacion | Parquet plano | Iceberg |
|-----------|--------------|--------|
| Lectura analitica | Excelente | Excelente |
| UPDATE de un registro | Imposible sin reescribir todo | Nativo (ACID) |
| DELETE de filas | Imposible sin reescribir todo | Nativo (ACID) |
| Ver estado anterior | Imposible | Time Travel con snapshots |
| Agregar columna | Requiere reescritura o nuevos archivos incompatibles | Schema Evolution nativa |
| Transacciones concurrentes | Sin garantias | Optimistic Concurrency Control |

Vamos a demostrar el problema primero.

In [ ]:
# Cargar el CSV de ventas (mismo dataset del Taller 4)
df = spark.read.csv("/home/iceberg/data/ventas.csv", header=True, inferSchema=True)
df = df.withColumn("total", F.col("cantidad") * F.col("precio_unitario"))

print(f"Registros cargados: {df.count()}")
df.printSchema()
df.show(5)

# Escribir como Parquet plano en MinIO
df.write.mode("overwrite").parquet("s3://warehouse/ventas_parquet/")
print("\nEscrito como Parquet plano en s3://warehouse/ventas_parquet/")
print()
print("=" * 60)
print("PROBLEMA con Parquet plano:")
print("  - Para actualizar 1 registro hay que reescribir TODO el archivo")
print("  - No hay control de versiones ni snapshots")
print("  - No hay garantias ACID en escrituras concurrentes")
print("  - Agregar una columna puede romper lectores anteriores")
print("="* 60)

In [ ]:
# Intentar un UPDATE sobre Parquet plano — esto no existe en la API de Spark DataFrames
# La unica forma es: leer TODO, filtrar, transformar, reescribir TODO

print("Simulando UPDATE en Parquet plano (la unica forma posible):")
print("  1. Leer TODOS los datos del archivo Parquet")
print("  2. Aplicar la transformacion en memoria")
print("  3. Reescribir TODOS los datos")
print()

df_parquet = spark.read.parquet("s3://warehouse/ventas_parquet/")
df_actualizado = df_parquet.withColumn(
    "precio_unitario",
    F.when(F.col("producto") == "Laptop", F.lit(1300000)).otherwise(F.col("precio_unitario"))
)
df_actualizado.write.mode("overwrite").parquet("s3://warehouse/ventas_parquet/")

print("UPDATE simulado completado — pero implicó reescribir los 50 registros")
print("En un dataset de 10TB esto seria catastrofico en tiempo y costo.")
print()
print("Con Iceberg, un UPDATE solo reescribe los archivos afectados")
print("y crea un nuevo snapshot apuntando a los archivos sin cambios.")

## Parte 2: Crear tabla Iceberg

Ahora creamos la misma tabla pero como tabla Iceberg en el catalogo REST. Iceberg va a:

1. Escribir los datos en MinIO como archivos Parquet (igual que antes)
2. Crear archivos de **metadatos** (manifest list, manifest files, table metadata JSON) que registran exactamente qué archivos forman cada snapshot
3. Registrar la tabla en el **catálogo REST**, que es la fuente de verdad sobre qué snapshots existen

Después de ejecutar las celdas, ve a **MinIO Console (http://localhost:9001)** y explora el bucket `warehouse`. Vas a ver dos carpetas:
- `warehouse/taller5/ventas/data/` — archivos Parquet con los datos
- `warehouse/taller5/ventas/metadata/` — archivos JSON con los snapshots y manifests

In [ ]:
# Limpiar si existe una ejecucion anterior
spark.sql("DROP TABLE IF EXISTS demo.taller5.ventas")
spark.sql("DROP NAMESPACE IF EXISTS demo.taller5")

# Crear el namespace (equivalente a un schema/database)
spark.sql("CREATE NAMESPACE IF NOT EXISTS demo.taller5")
print("Namespace demo.taller5 creado")

# Recargar el CSV (sin el total calculado, para mantener el schema limpio)
df_base = spark.read.csv("/home/iceberg/data/ventas.csv", header=True, inferSchema=True)
df_base = df_base.withColumn("total", F.col("cantidad") * F.col("precio_unitario"))

# Escribir como tabla Iceberg
df_base.writeTo("demo.taller5.ventas").create()
print("Tabla Iceberg demo.taller5.ventas creada")
print()

# Verificar
print("Primeros 5 registros de la tabla Iceberg:")
spark.sql("SELECT * FROM demo.taller5.ventas LIMIT 5").show()

print(f"Total de registros: {spark.sql('SELECT COUNT(*) as total FROM demo.taller5.ventas').collect()[0].total}")

In [ ]:
# Explorar los archivos que Iceberg creo en MinIO
print("=== Archivos de datos (Parquet) ===")
spark.sql("""
    SELECT file_path, file_size_in_bytes, record_count
    FROM demo.taller5.ventas.files
""").show(truncate=False)

print("=== Snapshots ===")
spark.sql("""
    SELECT snapshot_id, committed_at, operation, summary
    FROM demo.taller5.ventas.snapshots
""").show(truncate=False)

## Parte 3: ACID — UPDATE y DELETE

Iceberg soporta DML transaccional completo desde Spark SQL. Cada operacion crea un **nuevo snapshot** que apunta a:
- Los archivos de datos **sin cambios** del snapshot anterior (sin copiarlos)
- Los **nuevos archivos** con las filas actualizadas o insertadas
- Archivos de **delete** con referencias a las filas eliminadas

Este modelo se llama **Copy-on-Write** (CoW) o **Merge-on-Read** (MoR) segun la configuracion.

Despues de cada operacion, ve a MinIO y observa que se crean nuevos archivos en `metadata/` — el snapshot anterior sigue intacto.

In [ ]:
# UPDATE: corregir el precio de todos los Laptop
print("Precios de Laptop ANTES del UPDATE:")
spark.sql("""
    SELECT id, producto, precio_unitario
    FROM demo.taller5.ventas
    WHERE producto = 'Laptop'
""").show()

spark.sql("""
    UPDATE demo.taller5.ventas
    SET precio_unitario = 1300000
    WHERE producto = 'Laptop'
""")
print("UPDATE ejecutado — nuevo snapshot creado")

print("\nPrecios de Laptop DESPUES del UPDATE:")
spark.sql("""
    SELECT id, producto, precio_unitario
    FROM demo.taller5.ventas
    WHERE producto = 'Laptop'
""").show()

In [ ]:
# DELETE: eliminar ventas con cantidad = 1 y precio < 20000
# (ninguna en este dataset, pero el mecanismo es lo que importa)
print("Total de registros ANTES del DELETE:")
spark.sql("SELECT COUNT(*) as total FROM demo.taller5.ventas").show()

# Primero veamos cuantos registros cumplen la condicion
candidatos = spark.sql("""
    SELECT COUNT(*) as candidatos
    FROM demo.taller5.ventas
    WHERE cantidad = 1 AND precio_unitario < 20000
""").collect()[0].candidatos
print(f"Registros que cumplen la condicion de DELETE: {candidatos}")

spark.sql("""
    DELETE FROM demo.taller5.ventas
    WHERE cantidad = 1 AND precio_unitario < 20000
""")
print("DELETE ejecutado — nuevo snapshot creado")

print("\nTotal de registros DESPUES del DELETE:")
spark.sql("SELECT COUNT(*) as total FROM demo.taller5.ventas").show()

print("Snapshots acumulados hasta ahora:")
spark.sql("""
    SELECT snapshot_id, committed_at, operation
    FROM demo.taller5.ventas.history
""").show(truncate=False)

## Parte 4: Time Travel

Iceberg mantiene un **historial completo de snapshots**. Cada snapshot es un estado consistente de la tabla en un momento dado.

Time Travel permite:
- **Auditoría**: ver como estaban los datos en una fecha concreta
- **Reproducibilidad**: ejecutar el mismo query de ML sobre los datos que existian cuando se entrenó el modelo
- **Recuperacion de errores**: si alguien hizo un DELETE masivo accidental, puedes leer el snapshot anterior

Se accede con `VERSION AS OF <snapshot_id>` o `TIMESTAMP AS OF '<timestamp>'`.

In [ ]:
# Ver el historial completo de snapshots con operaciones
print("=== Historial completo de la tabla ===")
spark.sql("""
    SELECT snapshot_id, committed_at, operation
    FROM demo.taller5.ventas.history
""").show(truncate=False)

In [ ]:
# Leer el snapshot inicial (antes de UPDATE y DELETE)
history = spark.sql("""
    SELECT snapshot_id, committed_at, operation
    FROM demo.taller5.ventas.history
    ORDER BY committed_at ASC
""").collect()

first_snapshot = history[0].snapshot_id
latest_snapshot = history[-1].snapshot_id

print(f"Snapshot inicial (creacion de tabla): {first_snapshot}")
print(f"Snapshot actual (mas reciente):       {latest_snapshot}")
print()

# Contar en el snapshot inicial
count_inicial = spark.sql(f"""
    SELECT COUNT(*) as total
    FROM demo.taller5.ventas VERSION AS OF {first_snapshot}
""").collect()[0].total

# Contar en el snapshot actual
count_actual = spark.sql("""
    SELECT COUNT(*) as total FROM demo.taller5.ventas
""").collect()[0].total

print(f"Registros en snapshot inicial: {count_inicial}")
print(f"Registros en snapshot actual:  {count_actual}")
print()

# Ver los precios de Laptop en el snapshot inicial
print("Precios de Laptop en el snapshot INICIAL (antes del UPDATE):")
spark.sql(f"""
    SELECT id, producto, precio_unitario
    FROM demo.taller5.ventas VERSION AS OF {first_snapshot}
    WHERE producto = 'Laptop'
""").show()

print("Precios de Laptop en el estado ACTUAL (despues del UPDATE):")
spark.sql("""
    SELECT id, producto, precio_unitario
    FROM demo.taller5.ventas
    WHERE producto = 'Laptop'
""").show()

## Parte 5: Schema Evolution

En Parquet plano, si agregas una columna a un archivo nuevo y intentas leerlo junto con archivos viejos que no tienen esa columna, puedes obtener errores o comportamientos inesperados dependiendo del lector.

Iceberg maneja **Schema Evolution de forma nativa**:
- Las columnas nuevas aparecen como `NULL` en filas escritas antes de que existiera la columna
- Las columnas eliminadas dejan de ser visibles pero los datos fisicos no se borran
- Los archivos Parquet viejos **no se reescriben** — Iceberg resuelve el mapeo en tiempo de lectura

Esto es posible porque Iceberg usa **IDs internos** para las columnas, no nombres. Incluso si renombras una columna, Iceberg sabe que es la misma columna por su ID.

In [ ]:
# Ver schema actual
print("Schema ANTES de agregar columna:")
spark.sql("DESCRIBE TABLE demo.taller5.ventas").show()

# Agregar columna sin reescribir datos
spark.sql("ALTER TABLE demo.taller5.ventas ADD COLUMN descuento DOUBLE")
print("Columna 'descuento' agregada al schema")
print()

# Las filas anteriores muestran NULL en la columna nueva
print("Filas existentes con la nueva columna (muestra NULL — sin reescritura):")
spark.sql("""
    SELECT id, producto, precio_unitario, descuento
    FROM demo.taller5.ventas
    LIMIT 5
""").show()

print("La columna nueva aparece como NULL en filas anteriores — sin reescribir datos")

In [ ]:
# Insertar nuevas filas con la columna descuento poblada
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DoubleType, DateType
from datetime import date

nuevas_filas = spark.createDataFrame([
    (51, date(2024, 6, 1), "Laptop", "Electronica", 2, 1300000, "Bogota", 2600000, 0.10),
    (52, date(2024, 6, 5), "Smartphone", "Movil", 1, 1200000, "Medellin", 1200000, 0.05),
], ["id", "fecha", "producto", "categoria", "cantidad", "precio_unitario", "region", "total", "descuento"])

nuevas_filas.writeTo("demo.taller5.ventas").append()
print("Nuevas filas con descuento insertadas")

print("\nMix de filas antiguas (descuento=NULL) y nuevas (descuento poblado):")
spark.sql("""
    SELECT id, producto, precio_unitario, descuento
    FROM demo.taller5.ventas
    WHERE id >= 49
    ORDER BY id
""").show()

print(f"Total de registros finales: {spark.sql('SELECT COUNT(*) as total FROM demo.taller5.ventas').collect()[0].total}")

## Resumen y comparativa

### Lo que demostramos en este taller

| Capacidad | Como se hace en Iceberg | Por que importa |
|-----------|------------------------|----------------|
| **ACID UPDATE** | `UPDATE ... SET ... WHERE` | Corregir datos sin reescribir todo el dataset |
| **ACID DELETE** | `DELETE FROM ... WHERE` | Cumplimiento GDPR: borrar datos de un usuario |
| **Time Travel** | `VERSION AS OF <snapshot_id>` | Auditoría, reproducibilidad, recuperacion de errores |
| **Schema Evolution** | `ALTER TABLE ... ADD COLUMN` | Agregar features sin migrar datos historicos |

### Iceberg por debajo: tres capas

```
Capa 1 - Catalogo (iceberg-rest)
  └── Sabe cual es el "current snapshot" de cada tabla

Capa 2 - Metadata layer (JSON en MinIO /metadata/)
  ├── table-metadata-xxxx.json   → schema, propiedades, lista de snapshots
  ├── snap-xxxx.avro             → manifest list (apunta a manifests)
  └── manifest-xxxx.avro         → lista de archivos Parquet con estadisticas

Capa 3 - Data layer (Parquet en MinIO /data/)
  └── xxxx.parquet               → datos reales, sin cambios entre snapshots
```

### Preguntas de analisis

1. Despues de hacer 3 operaciones (CREATE, UPDATE, DELETE), ¿cuantos snapshots existen en `.history`? ¿Por que?

2. Si haces un DELETE masivo accidental en produccion (sin Iceberg), ¿como lo recuperas? ¿y con Iceberg?

3. Explica con tus palabras por que Schema Evolution en Iceberg **no requiere reescribir los archivos Parquet** existentes.

4. En el modelo Copy-on-Write de Iceberg, un UPDATE sobre 5 filas en un archivo de 1M filas, ¿cuantas filas se reescriben realmente?

5. ¿Que ventaja tiene usar MinIO (S3-compatible) en vez de HDFS para este tipo de arquitectura? ¿Que implicaciones tiene para el Taller 6 con Trino?

---

### Siguiente paso: Taller 6

En el **Taller 6** agregaremos **Trino** al stack para hacer consultas SQL federadas sobre las tablas Iceberg sin Spark — completando la arquitectura de un **Lakehouse moderno**.